<a href="https://colab.research.google.com/github/FrauMeO/OutskillC14GRP63/blob/main/rag_assignment_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Project - Build a Cursor Docs Chatbot

You just completed the teaching notebook where we built a RAG chatbot for Claude Code docs using OpenAI. Now it's your turn!

**Your mission:** Build the same RAG pipeline, but for [Cursor](https://docs.cursor.com) docs, using **Openrouter api**.

### What's different from the teaching notebook?

| Piece | Teaching notebook (OpenAI) | This project (OpenRouter) |
|---|---|---|
| Embeddings | `text-embedding-3-small` | Same model but using OpenRouter |
| LLM | `gpt-4o-mini` | Same model but accessed through OpenRouter |
| SDK | `openai` | Same! |
| Chunking | LangChain `RecursiveCharacterTextSplitter` | Same! |
| Vector DB | ChromaDB | Same! |

### How this notebook works

Most cells are ready to run. But at key steps — the ones that matter for understanding RAG — you'll see **TODO** markers where you need to fill in **1-2 lines of code**. Each TODO has a hint telling you exactly which method or value to use.

```python
# TODO: Create the splitter (hint: use RecursiveCharacterTextSplitter)
splitter = ?  # <-- you fill this in
```

**You are NOT writing functions from scratch.** The scaffolding is done — you just supply the critical lines that make the RAG pipeline work.

### If you plan to use Gemini directly, instead of OpenRouter or OpenAI. You can get your free Gemini API key through:

1. Go to https://aistudio.google.com/app/apikey
2. Click "Create API key"
3. Copy the key
4. In Colab: click the key icon in the left sidebar, add a secret named `GEMINI_API_KEY`

---

## Step 0: Setup

Just run these two cells. Nothing to change here.

In [ ]:
!pip install -q langchain openai langchain-community langchain-text-splitters chromadb httpx tqdm gradio

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception:
    pass

assert os.environ.get("OPENROUTER_API_KEY"), (
    "Set OPENROUTER_API_KEY in your environment or Colab secrets (left sidebar key icon)."
)
print("OPENROUTER_API_KEY is set. Ready to go!")

OPENROUTER_API_KEY is set. Ready to go!


---
## Step 1: Fetch the Cursor Docs

Unlike Claude Code (which has a single `llms-full.txt`), Cursor publishes an index file (`llms.txt`) that links to ~150 individual markdown pages. We need to:

1. Fetch the index
2. Extract all the `.md` page URLs
3. Download each page
4. Wrap each page as a LangChain `Document` object

Remember from the teaching notebook: a `Document` has `.page_content` (the text) and `.metadata` (title, source, etc.).

**Just run this cell** (the data fetching is done for you). (~20-30 seconds)

In [ ]:
import httpx
import time
from langchain_core.documents import Document

INDEX_URL = "https://cursor.com/llms.txt"
print(f"Fetching index {INDEX_URL}...")
index = httpx.get(INDEX_URL, timeout=60, follow_redirects=True).text

# Extract .md page URLs from the index
urls = []
seen = set()
for line in index.splitlines():
    if "https://" not in line or ".md" not in line:
        continue
    start = line.find("https://")
    end = line.find(".md", start)
    if start == -1 or end == -1:
        continue
    url = line[start : end + 3]
    if "cursor.com" in url and url not in seen:
        seen.add(url)
        urls.append(url)
print(f"Found {len(urls)} page URLs")

# Download each page and wrap as a LangChain Document
docs = []
for i, url in enumerate(urls, 1):
    try:
        page = httpx.get(url, timeout=60, follow_redirects=True).raise_for_status().text
    except httpx.HTTPError as e:
        print(f"  SKIP {url} ({e})")
        continue
    title = url.rsplit("/", 1)[-1].removesuffix(".md").replace("-", " ").title()
    docs.append(Document(
        page_content=f"# {title}\n\n{page.strip()}",
        metadata={"title": title, "source": url},
    ))
    if i % 20 == 0:
        print(f"  fetched {i}/{len(urls)}")
    time.sleep(0.05)

print(f"\nLoaded {len(docs)} documents.")
print(f"Example — title: '{docs[0].metadata['title']}'")
print(f"First 200 chars: {docs[0].page_content[:200]}...")

---
## Step 2: Chunk the Documents

Now we split the documents into smaller pieces. Remember from the teaching notebook:

- **Why chunk?** LLMs work better with focused, relevant paragraphs than entire pages
- **`RecursiveCharacterTextSplitter`** splits on paragraph breaks first, then sentences, then words
- **`chunk_size`** = max characters per chunk
- **`chunk_overlap`** = characters shared between adjacent chunks (so sentences at boundaries aren't lost)

### Your turn!

Fill in the TODO below to create the text splitter. Use the same settings we used in the teaching notebook:
- `chunk_size=512`
- `chunk_overlap=50`

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: Create the text splitter with chunk_size=512 and chunk_overlap=50
# Hint: RecursiveCharacterTextSplitter(chunk_size=???, chunk_overlap=???)
splitter = ??

# TODO: Split the docs into chunks
# Hint: the splitter has a method called split_documents() that takes a list of Documents
chunks = ??

print(f"{len(docs)} documents  -->  {len(chunks)} chunks")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"\n--- Example chunk ---")
print(f"Metadata: {chunks[0].metadata}")
print(f"Text: {chunks[0].page_content[:300]}...")

---
## Step 3: Embed & Store in ChromaDB

Now we embed each chunk into a vector and store it in ChromaDB.

**One difference from the teaching notebook:** Now, instead of directly using openai from Openai, since we are accessing OpenAI model through OpenRouter, we will need to pass the OpenRouter base api url.

### Your turn!

Fill in the TODO to add the chunks to the Chroma collection. Think about what data each chunk needs:
- An **ID** (unique identifier)
- A **document** (the text to embed)
- **Metadata** (title and source, for citations later)

In [ ]:
import chromadb
from chromadb import Documents, Embeddings, EmbeddingFunction
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from openai import OpenAI
from tqdm.auto import tqdm

CHROMA_DIR = "./chroma_db"
COLLECTION = "cursor_docs"
EMBED_MODEL = "openai/text-embedding-3-small"
BATCH_SIZE = 100


# Create (or recreate) the collection
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass

# We can safely use Chroma's built-in wrapper now
embed_fn = OpenAIEmbeddingFunction(
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",  # THIS IS THE EXTRA ADDITION AS COMPARED TO THE TEACHING NOTEBOOK
    model_name=EMBED_MODEL
)

# TODO: Create a Chroma collection to store our embeddings
# Hint: chroma_client has a method called create_collection()
#   Pass these arguments:
#     name=COLLECTION
#     embedding_function=embed_fn
#     metadata={"hnsw:space": "cosine"}   (tells Chroma to use cosine similarity)
collection = chroma_client.??()

# Add chunks to the vector DB
print(f"Embedding and storing {len(chunks)} chunks...\n")
for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
    batch = chunks[i : i + BATCH_SIZE]

    # TODO: Call collection.add() with the batch data
    # Hint: you need to complete two arguments:
    #   ids       = a list of unique string IDs (e.g., f"chunk-{i+j}" for each j)
    #   documents = a list of text strings (each chunk's .page_content)
    collection.add(
        ids=??,  # <-- TODO
        documents=??, # <-- TODO
        metadatas=[
            {
                "title": c.metadata.get("title", "Unknown"),
                "source": c.metadata.get("source", "Unknown"),
            }
            for c in batch
        ],
    )

print(f"\nDone! Collection has {collection.count()} embeddings.")

---
## Step 4:Retrieve

Time to search! Remember: this is pure vector similarity - no LLM yet.

The query gets embedded with the same model, and Chroma finds the `k` closest chunk vectors.

### Your turn!

Fill in the retrieve function. Chroma's collection has a `.query()` method.

In [ ]:
TOP_K = 5

def retrieve(query, k=TOP_K):
    """Search the vector DB for the top-k chunks most similar to the query."""
    # TODO: Call collection.query() to search for similar chunks
    # Hint: collection.query(query_texts=[query], n_results=???)
    return ??


# Test it with a Cursor-specific question
query = "How does Cursor's tab autocomplete work?"  # You can ask your own question here from cursor docs to test the retrieval
results = retrieve(query)

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    similarity = 1 - dist
    print(f"--- Rank {i}  (similarity: {similarity:.3f}) ---")
    print(f"Title:  {meta['title']}")
    print(f"Source: {meta['source']}")
    print(f"Text:   {doc[:250]}...\n")

---
## Step 5: Generate with LLM

Now the "G" in RAG. We take the retrieved chunks, put them in the prompt as context, and ask Gemini to answer.

**Gemini SDK difference:** If you plan to use Gemini, then instead of `openai_client.chat.completions.create()`, Gemini uses `client.models.generate_content()`. The system prompt goes in a `config` parameter instead of a separate message.

### Your turn!

Two TODOs here:
1. Write the system prompt — tell the LLM to only use the provided context, cite sources, and say "I don't know" when unsure
2. Complete the `ask()` function by calling retrieve + build_context + Gemini

In [ ]:
from openai import OpenAI

CHAT_MODEL = "gemini-2.5-flash"

# TODO: Write the system prompt.
# It should instruct the LLM to:
#   1. Answer ONLY using the provided context
#   2. Say "I don't know" if the answer isn't in the context
#   3. Cite source URLs using [1], [2], etc.
# Hint: look at the SYSTEM_PROMPT from the teaching notebook for inspiration
SYSTEM_PROMPT = """??"""

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)


def build_context(query, retrieved):
    """Format retrieved chunks into numbered context blocks."""
    docs = retrieved["documents"][0]
    metas = retrieved["metadatas"][0]
    blocks = [
        f"[{i}] Source: {m['source']}\n{d}"
        for i, (d, m) in enumerate(zip(docs, metas), start=1)
    ]
    context_str = "\n\n---\n\n".join(blocks)
    return f"Context:\n\n{context_str}\n\nQuestion: {query}", metas


def ask(query):
    """Full RAG pipeline: retrieve, build context, generate answer."""
    # TODO: Complete the RAG pipeline
    # Step 1: Retrieve the top-k chunks for the query
    # Hint: use the retrieve() function you wrote above
    retrieved = ??

    # Step 2: Build the context string (this is done for you)
    user_msg, metas = build_context(query, retrieved)

    # Step 3: Call OpenAI to generate the answer
    # Hint: client.chat.completions.create (check )
    resp = client.chat.completions.create(
        model=??,  # <-- TODO
        messages=??,  # <-- TODO
        temperature=??  # <-- TODO
    )

    return resp.text, metas


# Test it!
answer, sources = ask("What is agent mode in Cursor?")
print(answer)
print("\n--- Sources ---")
for i, s in enumerate(sources, 1):
    print(f"  [{i}] {s['title']}  —  {s['source']}")

---
## Step 6: Evaluate

Remember: you wouldn't ship code without tests, and you shouldn't ship a RAG app without evals.

We measure:
- **Retrieval hit-rate:** did the right chunks get retrieved?
- **Answer qualityL** LLM-as-judge scores 1-5

The eval set and judge are provided. But you need to fill in the retrieval hit check.

### Your turn!

Run the cell and see how your RAG app is performing.

In [ ]:
EVAL_SET = [
    {
        "question": "How does Cursor's tab autocomplete work?",
        "expected_source_contains": "tab",
        "expected_answer_gist": (
            "Cursor Tab is an autocomplete model that predicts multi-line edits "
            "across the file based on recent edits and cursor position; accept with Tab, reject with Esc."
        ),
    },
    {
        "question": "What is agent mode in Cursor?",
        "expected_source_contains": "agent",
        "expected_answer_gist": (
            "Agent mode lets Cursor autonomously make multi-file edits, run terminal commands, "
            "and use tools to complete a task end-to-end while the user reviews changes."
        ),
    },
    {
        "question": "How do I add an MCP server in Cursor?",
        "expected_source_contains": "mcp",
        "expected_answer_gist": (
            "Configure MCP servers in .cursor/mcp.json or via Settings -> MCP; "
            "specify command, args, and env; Cursor exposes the server's tools to the agent."
        ),
    },
    {
        "question": "What are Cursor rules and how do I use them?",
        "expected_source_contains": "rules",
        "expected_answer_gist": (
            "Rules are project- or user-level instructions in .cursor/rules/*.mdc "
            "loaded into the model context to enforce conventions or constraints."
        ),
    },
    {
        "question": "Which AI models does Cursor support?",
        "expected_source_contains": "models",
        "expected_answer_gist": (
            "Cursor supports multiple frontier models including Claude, GPT, Gemini, "
            "and Cursor's own models; selectable in the model picker."
        ),
    },
]

JUDGE_PROMPT = """You are scoring how well a generated answer matches an expected answer.

Question: {question}
Expected answer (gist): {expected}
Generated answer: {generated}

Score 1-5:
  1 = wrong or empty
  2 = partially relevant but missing key facts
  3 = roughly right but vague or incomplete
  4 = correct, covers the gist, minor omissions ok
  5 = correct, complete, and well cited

Reply with just the integer score, nothing else."""


def judge_answer(question, expected, generated):
    resp = gem.models.generate_content(
        model=CHAT_MODEL,
        contents=JUDGE_PROMPT.format(
            question=question, expected=expected, generated=generated
        ),
        config=types.GenerateContentConfig(temperature=0),
    )
    raw = (resp.text or "").strip()
    for tok in raw.split():
        try:
            return max(1, min(5, int(tok)))
        except ValueError:
            continue
    return 1


# Run the eval
print("Running evaluation...\n")
total_hits = 0
total_score = 0

for item in EVAL_SET:
    r = retrieve(item["question"], k=TOP_K)
    sources = [m["source"] for m in r["metadatas"][0]]
    hit = any(item["expected_source_contains"] in s for s in sources)
    total_hits += int(hit)

    answer, _ = ask(item["question"])
    score = judge_answer(item["question"], item["expected_answer_gist"], answer)
    total_score += score

    status = "HIT " if hit else "MISS"
    print(f"  [{status}] score={score}/5  {item['question']}")

n = len(EVAL_SET)
print(f"\n{'='*50}")
print(f"  Retrieval hit-rate:  {total_hits}/{n} = {total_hits/n:.0%}")
print(f"  Avg answer score:    {total_score/n:.1f} / 5.0")
print(f"{'='*50}")

---
## Step 7: Chatbot UI

Just run this cell — the Gradio chatbot uses the `ask()` function you already built above. Nothing to fill in here.

In [ ]:
import gradio as gr


def chatbot(message, history):
    """Handle a chat message: retrieve, generate, format with sources."""
    retrieved = retrieve(message, k=TOP_K)
    user_msg, metas = build_context(message, retrieved)
    resp = gem.models.generate_content(
        model=CHAT_MODEL,
        contents=user_msg,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT, temperature=0.1
        ),
    )
    answer = resp.text
    source_lines = [
        f"[{i}] {m['title']} — {m['source']}"
        for i, m in enumerate(metas, 1)
    ]
    return f"{answer}\n\n---\n**Sources retrieved:**\n" + "\n".join(source_lines)


gr.ChatInterface(
    fn=chatbot,
    title="Cursor Docs Chatbot (RAG)",
    description="Ask anything about Cursor. Powered by LangChain + ChromaDB + Gemini.",
    examples=[
        "How does Cursor tab autocomplete work?",
        "What is agent mode?",
        "How do I add an MCP server in Cursor?",
        "What are Cursor rules?",
    ],
).launch(share=False)

---
## Done!

If you can demo this chatbot and read your eval scoreboard, you've built first second RAG app.

### Recap of what YOU filled in

| Step | What you wrote | RAG concept |
|---|---|---|
| Chunking | `RecursiveCharacterTextSplitter(...)` | How documents get split into searchable pieces |
| Storing | `collection.add(ids, documents, metadatas)` | What data goes into the vector DB |
| Retrieving | `collection.query(query_texts, n_results)` | How similarity search works |
| System prompt | The grounding instructions | How to prevent hallucination |
| RAG pipeline | `retrieve → build_context → generate` | The full retrieve-then-generate flow |
| Eval | `any(keyword in source for source in sources)` | How to measure retrieval quality |

### Bonus challenges (if you have time)

1. **Add a PDF:** Use `PyPDFLoader` (from the teaching notebook) to load a Cursor-related PDF and add its chunks to the same collection
2. **Tune chunk_size:** Try 1000 or 3000 instead of 512 and re-run the eval and compare scores
3. **Add your own eval question:** Add a 6th item to `EVAL_SET` about a Cursor feature you care about
4. **Compare models:** Change `CHAT_MODEL` to a different Gemini model and see if scores change